# C1: choosing the next token

**Goal:** understand greedy, temperature, top-k and top-p sampling.

B8 produced logits. Now we choose a token from one `(vocab,)` row. C2 will put this choice inside a generation loop.

**Spec:** `sampling_probs(logits, temperature=1, top_k=0, top_p=1)` returns probabilities. `sample_token` returns a Python token ID and defaults to greedy (`temperature=0`). Temperature -> top-k -> top-p -> softmax -> random draw.

Our tiny vocabulary is Paris, London, Berlin, Rome, with **synthetic** probabilities `[0.60, 0.25, 0.10, 0.05]`. These are not TinyLlama measurements.

Run top to bottom in Colab on a T4. **No model weights are loaded**, so this lesson uses little memory.

In [1]:
import os, subprocess, sys
REPO_URL = "https://github.com/guptasameer112/tinyserve.git"
REPO_DIR = "/content/tinyserve"
if os.path.isdir(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)
print(subprocess.run(["git", "log", "--oneline", "-1"], check=True, capture_output=True, text=True).stdout)


2f943b7 C1: sampling probabilities, tests and Colab lesson



In [2]:
import importlib
import torch
import tinyserve.sampling
importlib.reload(tinyserve.sampling)
from tinyserve.sampling import sampling_probs, sample_token
from tinyserve.envinfo import collect_env, save_result
if not torch.cuda.is_available():
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then rerun.")
DEVICE = "cuda"
print(collect_env())


{'python': '3.13.16', 'platform': 'Linux-6.6.159+-x86_64-with-glibc2.39', 'git_sha': '2f943b7', 'gpu': 'Tesla T4', 'torch': '2.11.0+cu130', 'cuda': '13.0', 'transformers': '5.21.0'}


## 1. Test first

Tests define hand probabilities, temperature effects, filter order, ties, HF warper agreement, seeded draws and invalid inputs. A failed test stops the notebook. HF and CUDA checks should not be skipped here.

In [3]:
test_run = subprocess.run([sys.executable, "-m", "pytest", "tests/test_c1_sampling.py", "-v", "-ra"],
                          capture_output=True, text=True)
print(test_run.stdout)
print(test_run.stderr)
test_run.check_returncode()


============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/tinyserve
configfile: pyproject.toml
plugins: typeguard-4.6.0, langsmith-0.14.13, anyio-4.15.1, platformdirs-4.13.6
collecting ... collected 42 items

tests/test_c1_sampling.py::test_plain_softmax_matches_hand_probabilities PASSED [  2%]
tests/test_c1_sampling.py::test_temperature_squares_probabilities_at_half_temperature PASSED [  4%]
tests/test_c1_sampling.py::test_higher_temperature_flattens_distribution PASSED [  7%]
tests/test_c1_sampling.py::test_greedy_is_one_hot_lowest_index_on_tie_and_does_not_consume_rng PASSED [  9%]
tests/test_c1_sampling.py::test_top_k_two PASSED                         [ 11%]
tests/test_c1_sampling.py::test_top_k_keeps_threshold_ties PASSED        [ 14%]
tests/test_c1_sampling.py::test_top_k_larger_than_vocab_is_unfiltered PASSED [ 16%]
tests/test_c1_samp

## 2. Greedy versus temperature

Greedy always picks Paris. Temperature does not choose a fixed token; it changes the probabilities used for a random draw.

- `T=0`: our greedy convention.
- `T=0.5`: stronger preference for Paris.
- `T=1`: original distribution.
- `T=2`: more even distribution.

We divide **scores** by T before softmax. Since these scores are log probabilities, T=0.5 squares the probabilities and renormalizes them. Lower temperature does not guarantee a correct answer; higher temperature does not guarantee useful creativity.

In [4]:
token_names = ["Paris", "London", "Berlin", "Rome"]
toy_logits = torch.tensor([0.60, 0.25, 0.10, 0.05], device=DEVICE).log()
temperature_rows = []
for temperature in (0.0, 0.5, 1.0, 2.0):
    probabilities = sampling_probs(toy_logits, temperature=temperature)
    temperature_rows.append({"temperature": temperature, "probabilities": probabilities.tolist()})
    print(f"T={temperature}: " + ", ".join(f"{name} {probability:.1%}" for name, probability in zip(token_names, probabilities.tolist())))
print("greedy choice:", token_names[sample_token(toy_logits)])


T=0.0: Paris 100.0%, London 0.0%, Berlin 0.0%, Rome 0.0%
T=0.5: Paris 82.8%, London 14.4%, Berlin 2.3%, Rome 0.6%
T=1.0: Paris 60.0%, London 25.0%, Berlin 10.0%, Rome 5.0%
T=2.0: Paris 42.7%, London 27.6%, Berlin 17.4%, Rome 12.3%
greedy choice: Paris


## 3. Top-k versus top-p

**Top-k=2:** retain the two highest-score candidates, Paris and London.

**Top-p=0.8:** Paris alone contributes 60%, so include London to reach 85%. Retain both, then renormalize: Paris 60/85 = 70.6%, London 25/85 = 29.4%.

K is a rank cutoff; p adapts to probability mass. P does not mean the best token gets that probability.

Top-k runs before top-p, so top-p sees the already-filtered distribution. Top-k cutoff ties can retain more than k tokens. Top-p always keeps at least one token and follows HF's ascending-tail boundary convention.

In [5]:
filter_settings = [
    {"label": "unfiltered", "top_k": 0, "top_p": 1.0},
    {"label": "top-k=2", "top_k": 2, "top_p": 1.0},
    {"label": "top-p=0.8", "top_k": 0, "top_p": 0.8},
    {"label": "top-k=3 then top-p=0.8", "top_k": 3, "top_p": 0.8},
    {"label": "top-p=0.1", "top_k": 0, "top_p": 0.1},
]
filter_rows = []
for setting in filter_settings:
    probabilities = sampling_probs(toy_logits, top_k=setting["top_k"], top_p=setting["top_p"])
    filter_rows.append({**setting, "probabilities": probabilities.tolist()})
    print(setting["label"] + ": " + ", ".join(f"{name} {probability:.1%}" for name, probability in zip(token_names, probabilities.tolist())))


unfiltered: Paris 60.0%, London 25.0%, Berlin 10.0%, Rome 5.0%
top-k=2: Paris 70.6%, London 29.4%, Berlin 0.0%, Rome 0.0%
top-p=0.8: Paris 70.6%, London 29.4%, Berlin 0.0%, Rome 0.0%
top-k=3 then top-p=0.8: Paris 70.6%, London 29.4%, Berlin 0.0%, Rome 0.0%
top-p=0.1: Paris 100.0%, London 0.0%, Berlin 0.0%, Rome 0.0%


## 4. Random draws and reproducibility

The distribution stays fixed; each draw can differ. Recreate the same generator seed to replay the choices on this device/software. A GPU generator must match the GPU logits.

This repeatedly samples one unchanged logits row. It is **not text generation**: a real generation loop recalculates logits after every appended token.

In [6]:
sample_generator = torch.Generator(device=DEVICE).manual_seed(17)
chosen_ids = [sample_token(toy_logits, temperature=1.0, top_k=2, generator=sample_generator) for _ in range(20)]
repeat_generator = torch.Generator(device=DEVICE).manual_seed(17)
repeated_ids = [sample_token(toy_logits, temperature=1.0, top_k=2, generator=repeat_generator) for _ in range(20)]
assert chosen_ids == repeated_ids
assert set(chosen_ids).issubset({0, 1})
print("draws:", [token_names[token] for token in chosen_ids])
print("same-seed replay matches:", chosen_ids == repeated_ids)


draws: ['Paris', 'Paris', 'London', 'Paris', 'London', 'Paris', 'Paris', 'Paris', 'Paris', 'London', 'Paris', 'Paris', 'London', 'Paris', 'Paris', 'Paris', 'Paris', 'London', 'Paris', 'Paris']
same-seed replay matches: True


## 5. Frequency ranges across three seeded trials

Draw 4000 samples in each trial. Frequencies should be near 60%, 25%, 10%, 5%, not exactly equal. Compare against the actual probabilities and report the observed min/max per token across trials.

For 4000 draws, the largest standard error is about 0.008. Absolute tolerance 0.04 is about five standard errors. Seeds differ across trials, but all use the same distribution.

In [9]:
plain_probabilities = sampling_probs(toy_logits)
frequency_rows = []
for seed in (17, 18, 19):
    trial_generator = torch.Generator(device=DEVICE).manual_seed(seed)
    draws = torch.multinomial(plain_probabilities, 4000, replacement=True, generator=trial_generator)
    frequencies = torch.bincount(draws, minlength=4).float() / draws.numel()
    torch.testing.assert_close(frequencies, plain_probabilities, rtol=0, atol=0.04)
    frequency_rows.append({"seed": seed, "draws": draws.numel(), "frequencies": frequencies.tolist()})
frequency_tensor = torch.tensor([row["frequencies"] for row in frequency_rows], device=DEVICE)
frequency_ranges = []
for token_id, name in enumerate(token_names):
    minimum = frequency_tensor[:, token_id].min().item()
    maximum = frequency_tensor[:, token_id].max().item()
    frequency_ranges.append({"token": name, "min": minimum, "max": maximum})
    print(f"{name}: {minimum:.1%} .. {maximum:.1%}")


Paris: 59.0% .. 60.0%
London: 24.1% .. 26.2%
Berlin: 9.7% .. 10.8%
Rome: 5.1% .. 5.2%


## 6. Save evidence and three interview questions

1. **Temperature vs filters?** Temperature reshapes probabilities; top-k/top-p remove candidates.
2. **Top-k vs top-p?** K uses a fixed score-rank cutoff; p chooses candidates by cumulative probability mass.
3. **Does a seed reproduce results everywhere?** No: it is reproducible within the same device/software conditions.

Save with outputs and retain the printed JSON. There are no model tensors to free in this notebook.

**Next: C2 generation loop**, using our model and this sampler, with EOS and token-count stopping.

In [10]:
saved_path = save_result("C1_sampling", {
    "synthetic_vocabulary": token_names,
    "base_probabilities": plain_probabilities.tolist(),
    "temperature_comparison": temperature_rows,
    "filter_comparison": filter_rows,
    "seeded_sample_ids": chosen_ids,
    "same_seed_replay_equal": chosen_ids == repeated_ids,
    "frequency_trials": frequency_rows,
    "frequency_ranges": frequency_ranges,
    "pytest": {"returncode": test_run.returncode, "stdout": test_run.stdout, "stderr": test_run.stderr},
})
print(saved_path)
print(saved_path.read_text())


/content/tinyserve/results/C1_sampling.json
{
  "name": "C1_sampling",
  "timestamp_utc": "2026-10-31T06:55:36Z",
  "env": {
    "python": "3.13.16",
    "platform": "Linux-6.6.159+-x86_64-with-glibc2.39",
    "git_sha": "2f943b7",
    "gpu": "Tesla T4",
    "torch": "2.11.0+cu130",
    "cuda": "13.0",
    "transformers": "5.21.0"
  },
  "data": {
    "synthetic_vocabulary": [
      "Paris",
      "London",
      "Berlin",
      "Rome"
    ],
    "base_probabilities": [
      0.6000000238418579,
      0.25,
      0.09999999403953552,
      0.04999999701976776
    ],
    "temperature_comparison": [
      {
        "temperature": 0.0,
        "probabilities": [
          1.0,
          0.0,
          0.0,
          0.0
        ]
      },
      {
        "temperature": 0.5,
        "probabilities": [
          0.8275861740112305,
          0.14367814362049103,
          0.02298850566148758,
          0.005747126415371895
        ]
      },
      {
        "temperature": 1.0,
        "prob